In [1]:
using Gridap
using Gridap.Fields
using Gridap.Polynomials
using Gridap.ReferenceFEs
using LinearAlgebra
using FillArrays
using StaticArrays

In [2]:

# -----------------------------
# Material parameters
# -----------------------------
E = 119e3
ν = 0.3
C = (E / (1 - ν^2)) * [
    1.0     ν      0.0;
    ν       1.0    0.0;
    0.0     0.0  (1 - ν) / 2
]  # Plane stress constitutive matrix (3x3)

# -----------------------------
# Mesh definition
# -----------------------------
nodes = Point{2,Float64}[
  (0,0),(0,1),(0,2),
  (1,0),(1,1),(1,2),
  (2,0),(2,1),(2,2)
]
conn = [[1,2,4,5],[2,3,5,6],[4,5,7,8],[5,6,8,9]]
ncells = length(conn)

# -----------------------------
# Reference shape functions (bilinear)
# -----------------------------
filter(e,p) = true
ref_nodes = Point{2,Float64}[(0,0),(1,0),(0,1),(1,1)]
m = MonomialBasis{2}(Float64,1,filter)
l = LagrangianDofBasis(Float64,ref_nodes)
change = inv(evaluate(l,m))
s = linear_combination(change,m)  # shape functions (vector of 4 scalar fields)

# -----------------------------
# Reference quadrature rule (1-point)
# -----------------------------
q = Point{2,Float64}[(0.5,0.5)]
w = [1.0]

# -----------------------------
# Cell-wise shape functions
# -----------------------------
cell_s = Fill(s,ncells)
cell_q = Fill(q,ncells)
cell_w = Fill(w,ncells)

# -----------------------------
# Geometry mapping
# -----------------------------
cell_nodes = lazy_map(Broadcasting(Reindex(nodes)), conn)
cell_φ = lazy_map(linear_combination, cell_nodes, cell_s)
cell_Jt = lazy_map(∇, cell_φ)
cell_detJ = lazy_map(Broadcasting(det), cell_Jt)
cell_invJt = lazy_map(Operation(inv), cell_Jt)

# -----------------------------
# Gradients in physical space
# -----------------------------
cell_∇ref_s = lazy_map(Broadcasting(∇), cell_s)
cell_∇s = lazy_map(Broadcasting(Operation(⋅)), cell_invJt, cell_∇ref_s)

# -----------------------------
# Strain-displacement (B) matrix and element stiffness
# -----------------------------
function B_matrix(∇s_q)
  # ∇s_q: gradients of shape functions at a quadrature point (4 x 2)
  nshape = length(∇s_q)
  B = zeros(3, 2*nshape)
  for i in 1:nshape
    dNdx = ∇s_q[i]
    B[1,2i-1] = dNdx[1]      # ε_xx
    B[2,2i]   = dNdx[2]      # ε_yy
    B[3,2i-1] = dNdx[2]      # γ_xy
    B[3,2i]   = dNdx[1]
  end
  return B
end

# Loop over all cells
Ke_collection = Vector{Matrix{Float64}}(undef, ncells)
for (icell, (∇s_cell, Jt, detJ, wq)) in enumerate(zip(cell_∇s, cell_Jt, cell_detJ, cell_w))
  # One quadrature point (can generalize for more)
  ∇s_q = [evaluate(∇s_cell[i], q[1]) for i in 1:length(∇s_cell)]
  B = B_matrix(∇s_q)
  Jdet = abs(detJ(q[1]))
  Ke = B' * C * B * Jdet * wq[1]
  Ke_collection[icell] = Ke
end
# -----------------------------

In [3]:

C = (E / (1 - ν^2)) * [
    1.0  ν   0.0;
    ν    1.0 0.0;
    0.0  0.0 (1-ν)/2
]

# -----------------------------
# Mesh
# -----------------------------
L, W = 60.0, 20.0
partition = (Int(L*3), Int(W*3))
model = CartesianDiscreteModel((0.0,L,0.0,W), partition)

labels = get_face_labeling(model)
add_tag_from_tags!(labels, "left", [1,3,7])
add_tag_from_tags!(labels, "right", [2,4,8])




tria = Triangulation(model)
ncells = num_cells(tria)
cell_coords = get_cell_coordinates(tria)

# -----------------------------
# Shape functions (bilinear)
# -----------------------------
ref_nodes = Point{2,Float64}[(0,0),(1,0),(0,1),(1,1)]
filter(e,p) = true
m = MonomialBasis{2}(Float64,1,filter)
l = LagrangianDofBasis(Float64,ref_nodes)
change = inv(evaluate(l,m))
s = linear_combination(change,m)

# Quadrature (1-point)
q = [Point{2,Float64}((0.5,0.5))]
w = [1.0]

cell_s = Fill(s, ncells)
cell_q = Fill(q, ncells)
cell_w = Fill(w, ncells)

cell_φ = lazy_map(linear_combination, cell_coords, cell_s)
cell_Jt = lazy_map(∇, cell_φ)
cell_detJ = lazy_map(Broadcasting(det), cell_Jt)
cell_invJt = lazy_map(Operation(inv), cell_Jt)
cell_∇ref_s = lazy_map(Broadcasting(∇), cell_s)
cell_∇s = lazy_map(Broadcasting(Operation(⋅)), cell_invJt, cell_∇ref_s)

# -----------------------------
# B-matrix
# -----------------------------
function B_matrix(∇s_q)
    nshape = length(∇s_q)
    B = zeros(3,2*nshape)
    for i in 1:nshape
        dNdx = ∇s_q[i]
        B[1,2i-1] = dNdx[1]
        B[2,2i]   = dNdx[2]
        B[3,2i-1] = dNdx[2]
        B[3,2i]   = dNdx[1]
    end
    return B
end

# # -----------------------------
# # Element stiffness matrices
# # -----------------------------
# Ke_collection = Vector{Matrix{Float64}}(undef, ncells)
# for (icell, (∇s_cell, detJ, wq)) in enumerate(zip(cell_∇s, cell_detJ, cell_w))
#     ∇s_q = [evaluate(∇s_cell[i], q[1]) for i in 1:length(∇s_cell)]
#     B = B_matrix(∇s_q)
#     Ke_collection[icell] = B' * C * B * abs(detJ(q[1])) * wq[1]
# end
# -----------------------------
# Element stiffness function for a single cell
# -----------------------------
function Ke_func(∇s_cell, detJ_cell, wq)
    ∇s_q = [evaluate(∇s_cell[i], q[1]) for i in 1:length(∇s_cell)]
    B = B_matrix(∇s_q)
    return B' * C * B * abs(detJ_cell(q[1])) * wq[1]
end
# -----------------------------
# Compute all element stiffness matrices using lazy_map
# -----------------------------
Ke_collection = lazy_map(Ke_func, cell_∇s, cell_detJ, cell_w)
# cell_mat = lazy_map( integrate,cell_∇s∇st,cell_q,cell_w,cell_Jt)
# -----------------------------
# Global DOFs
# -----------------------------
n_nodes = num_nodes(model)
ndofs = 2*n_nodes
K_global = zeros(ndofs, ndofs)
F_global = zeros(ndofs)

# -----------------------------
# Assemble global stiffness
# -----------------------------
conn = get_cell_node_ids(tria)
for (icell, Ke) in enumerate(Ke_collection)
    nodes = conn[icell]
    for i in 1:4, j in 1:4
        K_global[2*nodes[i]-1, 2*nodes[j]-1] += Ke[2*i-1, 2*j-1]
        K_global[2*nodes[i]-1, 2*nodes[j]]   += Ke[2*i-1, 2*j]
        K_global[2*nodes[i],   2*nodes[j]-1] += Ke[2*i,   2*j-1]
        K_global[2*nodes[i],   2*nodes[j]]   += Ke[2*i,   2*j]
    end
end


UndefVarError: UndefVarError: `get_cell_node_ids` not defined

In [4]:
# -----------------------------
# Load vector (right edge)
# -----------------------------
Γ = BoundaryTriangulation(model, tags="right")
face_node_ids = get_cell_node_ids(Γ)
face_coords   = get_cell_coordinates(Γ)

# Linear 1D shape
ref_nodes_line = Point{1,Float64}[Point(0.0), Point(1.0)]
m_line = MonomialBasis{1}(Float64,1)
l_line = LagrangianDofBasis(Float64, ref_nodes_line)
change_line = inv(evaluate(l_line,m_line))
s_line = linear_combination(change_line,m_line)
q_line = [Point{1,Float64}((0.5,))]
w_line = [1.0]
t = SVector(0.0, -1.0)

for iface in 1:length(face_node_ids)
    nodes = face_node_ids[iface]
    X_face = face_coords[iface]
    N_vals = [evaluate(s_line[i], q_line[1]) for i in 1:2]
    J = norm(X_face[2] - X_face[1])
    Fe = zeros(8)
    for i in 1:4
        Fe[2i-1] = t[1] * N_vals[i] * J * w_line[1]
        Fe[2i]   = t[2] * N_vals[i] * J * w_line[1]
    end
    for i in 1:4
        dof_x = 2*nodes[i]-1
        dof_y = 2*nodes[i]
        F_global[dof_x] += Fe[2i-1]
        F_global[dof_y] += Fe[2i]
    end
end

# -----------------------------
# Apply Dirichlet BC (left edge fixed)
# -----------------------------
Γ_left = BoundaryTriangulation(model, tags="left")
left_nodes = unique(vcat(get_cell_node_ids(Γ_left)...))
fixed_dofs = vcat(2*left_nodes .- 1, 2*left_nodes)
free_dofs = setdiff(1:ndofs, fixed_dofs)

K_ff = K_global[free_dofs, free_dofs]
F_f  = F_global[free_dofs]

# -----------------------------
# Solve
# -----------------------------
U_low_API = zeros(ndofs)
U_low_API[free_dofs] = K_ff \ F_f;


UndefVarError: UndefVarError: `get_cell_node_ids` not defined

In [5]:
# -----------------------------
L, W = 60.0, 20.0
partition = (Int(L*3), Int(W*3))
model = CartesianDiscreteModel((0.0,L,0.0,W), partition)

labels = get_face_labeling(model)
add_tag_from_tags!(labels, "left", [1,3,7])
add_tag_from_tags!(labels, "right", [2,4,8])

degree = 1
Ω = Triangulation(model)
dΩ = Measure(Ω, degree)
Γ = BoundaryTriangulation(model, tags="right")
dΓ = Measure(Γ, degree)

# -----------------------------
# FE space
# -----------------------------
order = 1
reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
Vh = TestFESpace(Ω, reffe; conformity=:H1, dirichlet_tags="left")
Uh = TrialFESpace(Vh)
# -----------------------------
# Plane stress constitutive matrix (component-wise)
# -----------------------------
# 2D plane stress Lamé parameters
λ = E*ν/(1-ν^2)
μ = E/(2*(1+ν))
# Create linear elasticity tensor (2D plane stress)
# -----------------------------
# Weak forms
# -----------------------------
# Weak forms
a(u,v) = ∫(
    λ * tr(ε(u)) * tr(ε(v)) + 2*μ * ε(u) ⊙ ε(v)
) * dΩ
Fvec(x) = VectorValue(0.0, -1.0)
l2(v) = ∫(Fvec ⋅ v) * dΓ
# -----------------------------
# Solve
# -----------------------------
op = AffineFEOperator(a, l2, Uh, Vh)
uh_high = solve(op)

SingleFieldFEFunction():
 num_cells: 10800
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 14806609165770092319

In [6]:
U_high_API = get_free_dof_values(uh_high) # displacement vector
println("Norm of uh in high API = ", norm(U_high_API), "\n")
println("Norm of uh in high API = ", norm(U_low_API))

Norm of uh in high API = 1.035825683671591



UndefVarError: UndefVarError: `U_low_API` not defined